In [1]:
# Imports
import numpy as np
import time 
import os
import h5py
import requests

# Import cyborgdb
import cyborgdb_core as cyborgdb

In [2]:
# Download test dataset
dataset = "sift-128-euclidean"
dataset_location = f'{dataset}.hdf5'
dataset_url = f'https://ann-benchmarks.com/{dataset}.hdf5'

# Check if the dataset exists, download if it doesn't
if not os.path.exists(dataset_location):
    print(f"Dataset not found. Downloading from {dataset_url}...")
    try:
        # Simple download with requests
        r = requests.get(dataset_url, stream=True)
        r.raise_for_status()  # Raise an exception for HTTP errors
        
        with open(dataset_location, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                
        print(f"Dataset downloaded to {dataset_location}")
    except Exception as e:
        print(f"Error downloading: {e}")
        print(f"Consider manually downloading the file from {dataset_url}")
        raise
else:
    print(f"Dataset already exists at {dataset_location}")

# Load the dataset
with h5py.File(dataset_location, 'r') as file:
    train = np.array(file['train'], dtype=np.float32)
    test = np.array(file['test'], dtype=np.float32)
    neighbors = np.array(file['neighbors'], dtype=np.int32)

num_vectors = train.shape[0]
dimension = train.shape[1]
num_queries = test.shape[0]

Dataset already exists at sift-128-euclidean.hdf5


In [3]:
# Location configuration (in-memory for this example)
backing_store = cyborgdb.DBConfig(location='memory')

# Uncomment the following line to use Redis
backing_store = cyborgdb.DBConfig(location='redis', table_name='cyborgdb_example', connection_string='host:127.0.0.1,port:6379,db:0')

# Index configuration (using IVFFlat)
n_lists = 4096
index_config = cyborgdb.IndexIVFFlat(dimension, n_lists)

# Setup the CVS index
client = cyborgdb.Client(
    index_location=backing_store,
    config_location=backing_store,
    items_location=backing_store
)

# Dummy index name and key
index_name = "memory_example_index"
index_key = bytes([1] * 32)

# Create the index
index = client.create_index(index_name, index_key, index_config)

In [7]:
# Create a list of items containing the vectors and their IDs
items = []
for i in range(num_vectors):
    items.append({
        'id': str(i),
        'vector': train[i]
    })

# Upsert untrained
start = time.time()
index.upsert(items)
print(f"Upserted {len(items)} vectors in {time.time() - start:.2f} seconds")

Upserted 1000000 vectors in 47.13 seconds


In [8]:
ids = np.arange(num_vectors).astype(str)

# Upsert NumPy
start = time.time()
index.upsert(ids, train)
print(f"Upserted {len(ids)} vectors in {time.time() - start:.2f} seconds")

Upserted 1000000 vectors in 43.57 seconds


In [ ]:
# Untrained query

num_queries = 1000
initial_queries = test[:num_queries]
initial_neighbors = neighbors[:num_queries]

start = time.time()
results = index.query(initial_queries, top_k=100, n_probes=1)
end = time.time()

# Parse results to extract IDs from the returned dictionaries
result_ids = [
    [int(res["id"]) for res in query_results] for query_results in results
]

# Convert the results to numpy array
result_ids = np.array(result_ids)
if initial_neighbors.shape != result_ids.shape:
    raise ValueError(f"The shapes of the neighbors and results do not match: {initial_neighbors.shape} != {result_ids.shape}")

# Compute the recall using the neighbors
recall = np.zeros(num_queries)
for i in range(num_queries):
    recall[i] = len(np.intersect1d(initial_neighbors[i], result_ids[i])) / len(initial_neighbors[i])

print(f"Queried {num_queries} vectors in {end - start:.2f} seconds")
print(f"QPS: {num_queries / (end - start):.2f}")
print(f"Mean recall: {recall.mean() * 100:.2f}%")

Queries may be slow; consider calling `TrainIndex()` to speed up queries.


Queried 1000 vectors in 11.52 seconds
QPS: 86.83
Mean recall: 99.99%


In [ ]:
# Train index for faster queries

start = time.time()
index.train()
print(f"Trained index with {len(items)} vectors in {time.time() - start:.2f} seconds")

Trained index with 1000000 vectors in 31.03 seconds


In [ ]:
# Trained query

# Query the test set
start = time.time()
results = index.query(initial_queries, top_k=100, n_probes=32)
end = time.time()

# Parse results to extract IDs from the returned dictionaries
result_ids = [
    [int(res["id"]) for res in query_results] for query_results in results
]

# Convert the results to numpy array
result_ids = np.array(result_ids)
if initial_neighbors.shape != result_ids.shape:
    raise ValueError(f"The shapes of the neighbors and results do not match: {initial_neighbors.shape} != {result_ids.shape}")

# Compute the recall using the neighbors
recall = np.zeros(num_queries)
for i in range(num_queries):
    recall[i] = len(np.intersect1d(initial_neighbors[i], result_ids[i])) / len(initial_neighbors[i])

print(f"Queried {initial_queries.shape[0]} vectors in {end - start:.2f} seconds")
print(f"QPS: {num_queries / (end - start):.2f}")
print(f"Mean recall: {recall.mean() * 100:.2f}%")

Queried 1000 vectors in 3.24 seconds
QPS: 308.29
Mean recall: 88.66%


In [ ]:
# Cleanup

index.delete_index()